## setup

In [75]:
from typing import List, Callable
from pydantic import BaseModel, Field

import os, requests, json, gc, rich
import pandas as pd
import numpy as np

from tqdm import tqdm

import faiss

from openai import OpenAI
from openai.types.chat import ParsedChatCompletion

In [20]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

FCLIP_API_TOKEN = os.getenv("HF_API_TOKEN")
FCLIP_API_ENDPOINT = "https://precove-fclip-back3.hf.space/encode_texts"

BATCH_SIZE = 128
LOAD_FAISS_INDEX = True

In [3]:
client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## utils

In [4]:
def create_embeddings_openai(texts: List[str]) -> List[List[float]]:
    response = client_openai.embeddings.create(
        input=texts,
        model=OPENAI_EMBEDDING_MODEL,
    )
    
    return [item.embedding for item in response.data]

In [5]:
def create_embeddings_fclip(texts: List[str]) -> List[List[float]]:
    headers = {
        "Authorization": f"Bearer {FCLIP_API_TOKEN}",
        "Content-Type": "application/json",
    }
    
    payload = json.dumps({"texts": texts})

    response = requests.request(
        method="POST",
        url=FCLIP_API_ENDPOINT,
        headers=headers,
        data=payload,
    )

    if response.ok:
        return response.json()["embeddings"]

    return None

In [6]:
def create_embeddings(texts: List[str], embed_func: Callable, batch_size: int) -> np.ndarray:
    n, n_success, all_embeddings = 0, 0, []
    loop = tqdm(iterable=range(0, len(texts), batch_size))

    for i in loop:
        n += 1

        try:
            batch = texts[i : i + batch_size]
            batch_embeddings = embed_func(batch)

            if batch_embeddings is not None:
                all_embeddings.extend(batch_embeddings)
                n_success += 1
        
        except Exception as e:
            loop.set_description(str(e))
        
        success_rate = n_success / n
        loop.set_description(f"{success_rate=:.2f}")

    embeddings = np.array(all_embeddings, dtype=np.float32)

    gc.collect()
    del all_embeddings

    return embeddings

In [18]:
def create_np_embedding(text: str, embed_func: Callable) -> np.ndarray:
    embedding = embed_func([text])[0]
    np_embedding = np.array(embedding, dtype=np.float32).reshape(1, -1)
    faiss.normalize_L2(np_embedding)

    return np_embedding

In [26]:
def search(
    embedding: np.ndarray,
    index: faiss.IndexFlatIP,
    dataset: pd.DataFrame,
    top_k: int
) -> pd.DataFrame:
    scores, indices = index.search(embedding, k=top_k)
    idx = indices[0]

    results = dataset.iloc[idx][["originalTitle", "longDescription"]].copy()
    results["score"] = scores[0]

    return results

In [76]:
def display_search_results(results: pd.DataFrame) -> None:
    rank = 0

    for row_number, row in results.iterrows():
        msg = (
            f"Rank: {rank}\n"
            f"Row: {row_number}\n"
            f"Title: {row['originalTitle']}\n"
            f"Description: {row['longDescription']}\n"
            f"Score: {row['score']:.3f}\n"
        )

        rich.print(msg)
        rank += 1

In [64]:
class OpenAIAgent:
    def __init__(
        self, 
        system_prompt: str,
        output_schema: BaseModel,
        model: str = "gpt-5-mini", 
        temperature: float = 1.
    ):
        self.system_prompt = system_prompt
        self.output_schema = output_schema
        self.model = model
        self.temperature = temperature
        
    def generate(self, text: str) -> ParsedChatCompletion:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": text}
        ]
        
        return client_openai.beta.chat.completions.parse(
            model=self.model,
            messages=messages,
            response_format=self.output_schema,
            temperature=self.temperature
        )

    def parse(self, response: ParsedChatCompletion) -> BaseModel:
        return response.choices[0].message.parsed       

## dataset

In [7]:
df = pd.read_csv("data/joko_products.csv")
print(df.shape)
df.head()

(17062, 2)


,originalTitle,longDescription
0,Light Blue Wash Fray Waistband Low Waist Strai...,Stay on trend with the light blue wash fray wa...
1,Dark Chocolate Cinched Long Sleeve Denim Jacket,Enhance your aesthetic in this dark chocolate ...
2,Black Diamante Detail Oversized Blazer Dress,We're all about the glam vibes this season and...
3,Petite Black Snatched Sculpt Strappy Maxi Dress,Master the minimalist mood with this black str...
4,Indigo Layered Exposed Pocket Wide Leg Jeans,"Consider these indigo, wide-leg jeans a master..."


In [8]:
df["text_openai"] = (
    df["originalTitle"].fillna("") + " " +
    df["longDescription"].fillna("")
).str.strip()

In [9]:
# we only use title since FashionCLIP is limited to 77 tokens
# and has been trained to map pixels to visual keywords that strictly describe clothing

df["text_fclip"] = df["originalTitle"].fillna("")

## embeddings

In [10]:
texts_openai = df["text_openai"].tolist()

In [11]:
embeddings_openai = create_embeddings(
    texts=texts_openai,
    embed_func=create_embeddings_openai,
    batch_size=BATCH_SIZE
)

success_rate=1.00: 100%|██████████| 134/134 [04:40<00:00,  2.10s/it]


In [12]:
texts_fclip = df["text_fclip"].tolist()

In [13]:
# takes some time to run since FashionCLIP encoder is hosted on Hugging Face CPU Basic space

embeddings_fclip = create_embeddings(
    texts=texts_fclip,
    embed_func=create_embeddings_fclip,
    batch_size=BATCH_SIZE
)

success_rate=1.00: 100%|██████████| 134/134 [15:02<00:00,  6.74s/it]


## `FAISS` index

In [21]:
if LOAD_FAISS_INDEX:
    index_openai = faiss.read_index("data/index_openai.faiss")

else:
    faiss.normalize_L2(embeddings_openai)

    dim_openai = embeddings_openai.shape[1]
    index_openai = faiss.IndexFlatIP(dim_openai)
    index_openai.add(embeddings_openai)

    faiss.write_index(index_openai, "data/index_openai.faiss")

print(f"OpenAI index: {index_openai.ntotal} vectors, dim={index_openai.d}")

OpenAI index: 17062 vectors, dim=1536


In [22]:
if LOAD_FAISS_INDEX:
    index_fclip = faiss.read_index("data/index_fclip.faiss")

else:
    faiss.normalize_L2(embeddings_fclip)

    dim_fclip = embeddings_fclip.shape[1]
    index_fclip = faiss.IndexFlatIP(dim_fclip)
    index_fclip.add(embeddings_fclip)

    faiss.write_index(index_fclip, "data/index_fclip.faiss")

print(f"FCLIP index:  {index_fclip.ntotal} vectors, dim={index_fclip.d}")

FCLIP index:  17062 vectors, dim=512


## search

In [77]:
query = "Ripped jeans with strass"

# "I want to take up yoga, what can I buy?"
# "Ripped jeans with strass"
# "Clothes for summer"
# "Outfit for attending a wedding"
# "Sports clothes for running"

In [78]:
embedding_openai = create_np_embedding(
    text=query, embed_func=create_embeddings_openai
)

results_openai = search(
    embedding=embedding_openai,
    index=index_openai,
    dataset=df,
    top_k=5
)

display_search_results(results_openai)

Rank: 0
Row: 2655
Title: Washed Black Extreme Distressed Sequin Panel Straight Leg Jeans
Description: Turn heads for all the right reasons in these washed black extreme distressed sequin panel straight 
leg jeans. Brought to you in a washed black hue material with an extreme distressed detail and sequin panel design,
these straight leg jeans are a yes from us. Team with the matching top, buckle heels and simple accessories for a 
statement fit. Length approx 81cm/32" (Based on a sample size UK 8) Model wears size UK 8/ EU 36/ AUS 8/ US 4
Score: 0.519

Rank: 1
Row: 12755
Title: Ecru Studded High Waist Jeans
Description: Keep it striking with the ecru studded high waist jeans. Made from an ecru denim material, they 
feature studded detailing, a high waist fit, and a straight leg cut. Style with a vest top, slip on mules and a 
clutch bag for timeless sophistication.
Score: 0.516

Rank: 2
Row: 2656
Title: Washed Stone Frayed Striped Seam Wide Leg Jeans
Description: Introducing your new style staple with these washed stone frayed striped seam wide leg jeans. Brought 
to you in a washed stone material with a frayed design, striped seam detail and wide leg fit, these jeans are a 
must have. How can you resist Style with a white tee, fresh kicks and simple accessories for a look like no other. 
Length approx 78cm/30.5" (Based on a sample size UK 8) Model wears size UK 8/ EU 36/ AUS 8/ US 4
Score: 0.503

Rank: 3
Row: 741
Title: Washed Grey Ruched Straight Leg Denim Jeans
Description: Opt for edge with the washed grey ruched straight leg denim jeans. Made from a washed grey denim 
material, they feature ruched detailing, a straight leg cut, and a relaxed fit. Pair with kitten heels and gold 
earrings for timeless chic.
Score: 0.493

Rank: 4
Row: 1985
Title: Shape Light Blue Acid Wash Denim Foldover Waist Ripped Wide Leg Jeans
Description: Designed with hourglass figures in mind these curve enhancing tweaks do the most (for both your figure
and confidence). PLT Shape looks good on you. Deconstructed denim reimagined for the discerning eye. These wide-leg
jeans offer an updated take on a classic silhouette, crafted from light blue acid wash denim with a contemporary 
foldover waist detail. Intentionally placed rips lend a touch of rebellious spirit, balancing the elevated design. 
Style this statement piece with a streamlined knit top and barely-there heels for a considered ensemble, or for an 
everyday fit, style with your favorite sneakers and pair with one of our skirts for a fresh take on denim dressing.
Length approx 81cm/32" | Shape Light Blue Acid Wash Denim Foldover Waist Ripped Wide Leg Jeans
Score: 0.486

In [79]:
embedding_fclip = create_np_embedding(
    text=query, embed_func=create_embeddings_fclip
)

results_fclip = search(
    embedding=embedding_fclip,
    index=index_fclip,
    dataset=df,
    top_k=5
)

display_search_results(results_fclip)

Rank: 0
Row: 5012
Title: Washed Stone Mid Rise Straight Leg Jeans
Description: Madre from a washed stone material with a must have low rise fit and a flattering straight leg design.
Take your look to new heights and pair these washed stone mid rise straight leg jeans with the matching long sleeve
jeacket and a pair of comfy sneakers to feel confident this season.
Score: 0.748

Rank: 1
Row: 7044
Title: Washed Stone Low Rise Wide Leg Jeans
Description: Keep it casual with the washed stone low rise wide leg jeans. Featuring a relaxed fit, low-rise waist 
and soft washed finish, they re your new everyday essential. Style with a baby tee or cropped blazer for off-duty 
cool.
Score: 0.747

Rank: 2
Row: 5335
Title: Washed Stone Mid Waist Wide Leg Jeans
Description: Define your denim aesthetic with a fresh perspective. The washed stone mid waist wide leg jeans offer 
a relaxed yet refined shape, featuring a clean mid-rise waist and an effortlessly flowing wide leg cut that feels 
directional. Classic denim details like front and back pockets, belt loops, and a button-fly fastening complete 
this essential style. Pair these jeans with a sharp blazer and tailored top for an elevated day look or a simple 
Vest and sneakers for off-duty cool.
Score: 0.744

Rank: 3
Row: 165
Title: Vintage Wash Split Hem Jeans
Description: Doll these jeans are perfect for your new-season wardrobe. Featuring a vintage wash denim material 
with split hem design and flattering fit. For a look that everyone will be obsessing over, style with a basic 
bodysuit, simple accessories, and mules. Length approx 82cm/32" (Based on a sample size UK 8) Model wears size UK 
10/ EU 38/ AUS 10/ US 6 Model Height - 5ft 5" | Vintage Wash Split Hem Jeans
Score: 0.732

Rank: 4
Row: 12646
Title: Light Stone Dip Front Straight Leg Jeans
Description: Stay modern in the light stone dip front straight leg jeans. Made from a light stone denim material, 
they feature a dip front waist, straight leg cut, and a flattering fit. Style with the matching jacket, slip on 
mules and a clutch bag for timeless appeal.
Score: 0.726

## evaluation

In [54]:
class QueryProductEval(BaseModel):
    reasoning: str = Field(
        description=(
            "A short, one-to-two sentence explanation "
            "of why this score was assigned based on the grading criteria."
        )
    )
    score: int = Field(
        description="Relevance score from 1 to 5.",
        ge=1, le=5
    )

In [55]:
QUERY_PRODUCT_EVAL_SYSTEM_PROMPT = """You are an expert e-commerce search quality evaluator.
Your task is to judge how relevant a retrieved product is to a user's search query.

# Grading Scale (1-5):
1 - Completely Irrelevant: The product has nothing to do with the query (e.g., query asks for shoes, product is a hat).
2 - Poor Match: Shares a vague category but misses crucial user constraints like gender, specific style, or color.
3 - Fair Match: The product is the right broad type, but misses a secondary but important detail requested by the user.
4 - Good Match: Highly relevant. Fits the core intent and most attributes. Might have a very minor mismatch (e.g., slightly different brand or shade).
5 - Perfect Match: Exact intent. Matches all explicit constraints (type, color, style, brand, usage) flawlessly.

Analyze the query and the product, write a brief reasoning, and then assign the score."""


In [57]:
def create_eval_user_prompt(query: str, product_title: str, product_description: str) -> str:
    return f"""
User Query: "{query}"

Retrieved Product Title: "{product_title}"
Retrieved Product Description: "{product_description}"

Evaluate the relevance."""

In [ ]:
agent_query_product = OpenAIAgent(
    system_prompt=QUERY_PRODUCT_EVAL_SYSTEM_PROMPT,
    output_schema=QueryProductEval,
    model="gpt-5-mini",
)

In [92]:
idx = 3
product_openai = results_openai.iloc[idx]
product_fclip = results_fclip.iloc[idx]

In [ ]:
user_prompt_openai = create_eval_user_prompt(
    query=query,
    product_title=product_openai["originalTitle"],
    product_description=product_openai["longDescription"]
)

rich.print(user_prompt_openai)

response_openai = agent_query_product.generate(text=user_prompt_openai)
response_parsed_openai = agent_query_product.parse(response_openai)

rich.print(response_parsed_openai.score)
rich.print(response_parsed_openai.reasoning)

User Query: "Ripped jeans with strass"

Retrieved Product Title: "Washed Grey Ruched Straight Leg Denim Jeans"
Retrieved Product Description: "Opt for edge with the washed grey ruched straight leg denim jeans. Made from a 
washed grey denim material, they feature ruched detailing, a straight leg cut, and a relaxed fit. Pair with kitten 
heels and gold earrings for timeless chic."

Evaluate the relevance.

2

The item is a pair of jeans (correct broad category) but the listing mentions ruched detailing only and does not 
include any ripped/distressed features or strass/rhinestone embellishments, so it fails to meet the user's main 
constraints.

In [ ]:
user_prompt_fclip = create_eval_user_prompt(
    query=query,
    product_title=product_fclip["originalTitle"],
    product_description=product_fclip["longDescription"]
)

rich.print(user_prompt_fclip)

response_fclip = agent_query_product.generate(text=user_prompt_fclip)
response_parsed_fclip = agent_query_product.parse(response_fclip)

rich.print(response_parsed_fclip.score)
rich.print(response_parsed_fclip.reasoning)

User Query: "Ripped jeans with strass"

Retrieved Product Title: "Vintage Wash Split Hem Jeans"
Retrieved Product Description: "Doll these jeans are perfect for your new-season wardrobe. Featuring a vintage wash
denim material with split hem design and flattering fit. For a look that everyone will be obsessing over, style 
with a basic bodysuit, simple accessories, and mules. Length approx 82cm/32" (Based on a sample size UK 8) Model 
wears size UK 10/ EU 38/ AUS 10/ US 6 Model Height - 5ft 5" | Vintage Wash Split Hem Jeans"

Evaluate the relevance.

2

The item is jeans (broad category match) but the listing describes a vintage wash and split hem with no 
ripped/distressed details or any strass/rhinestone embellishment, so it misses the user's key constraints.